# Q0 乱码是怎么产生的

In [ ]:
def save_to_computer(text):
    return text.encode("utf-8")

def read(text):
    return text.decode("gb18030")

def show_hex(text):
    # return " ".join(f"{ord(c):02x}" for c in text.hex())
    return text.hex(" ").upper()

In [ ]:
text = "你好"
print(text)

In [ ]:
stored_text = save_to_computer(text)
print(stored_text)

In [ ]:
# beauty print
print(show_hex(stored_text))

In [ ]:
fetched_text = read(stored_text)
print(fetched_text)

# Q3 ASCII：7 bit 与 1 byte

In [ ]:
ch = "A"
print(ch)

In [ ]:
# ord() returns the integer Unicode code point of a single character
code = ord(ch)
print(code)

In [ ]:
ascii_byte = ch.encode("ascii")[0]

print(f"Character        : {ch}")
print(f"Decimal code     : {code}")
print(f"Hex code         : 0x{code:02X}")
print(f"ASCII 7-bit code : {code:07b}")
print(f"Stored in 1 byte : {ascii_byte:08b}")
print(f"MSB              : {(ascii_byte >> 7) & 1}")

# Q6–Q7 Plan B：GB18030 机内码

In [ ]:
def bits(byte):
    return f"{byte:08b}"

def msb(byte):
    return (byte >> 7) & 1

In [ ]:
text = "中"
print(text.encode("gb18030").hex(" ").upper())

In [ ]:
text = "A"
print(text.encode("gb18030").hex(" ").upper())

In [ ]:
text = "B"
print(text.encode("gb18030").hex(" ").upper())

In [ ]:
text = "C"
print(text.encode("gb18030").hex(" ").upper())

In [ ]:
text = "A中B国C文"
print(text.encode("gb18030").hex(" ").upper())

In [ ]:
print("Text:", text)
print("GB18030 bytes:", text.encode("gb18030").hex(" ").upper())
print()
print(f"{'字符':<4}{'Hex':<10}{'Bytes':<8}{'Binary':<22}{'MSB'}")
print("-" * 60)

for ch in text:
    raw = ch.encode("gb18030")
    hex_bytes = " ".join(f"{b:02X}" for b in raw)
    binary = " ".join(bits(b) for b in raw)
    msbs = " ".join(str(msb(b)) for b in raw)
    print(f"{ch:<5}{hex_bytes:<13}{len(raw):<8}{binary:<22}{msbs}")

结论只限定在 GB2312 机内码模型。

---

# Q8 同一个汉字：GB18030 vs UTF-8

In [ ]:
ch = "中"
for encoding in ["gb2312", "utf-8"]:
    raw = ch.encode(encoding)
    print(f"{encoding:<8} bytes = {raw.hex(' ').upper():<10} length = {len(raw)}")

In [ ]:
text = "A中B国C文"
for encoding in ["gb2312", "utf-8"]:
    raw = text.encode(encoding)
    print(f"{encoding:<8}: {raw.hex(' ').upper()}")

结论：字节长度取决于编码方案。

---

# Q9 英文输入与中文输入 {#q9-demo}

In [ ]:
from IPython.display import display, HTML

english = [
    ("键盘输入", "按下 A"),
    ("字符确定", "字符 A"),
    ("字形信息", "font → glyph A"),
    ("显示", "pixels → A"),
]

chinese = [
    ("键盘输入", "按下 n、i"),
    ("输入码", "ni"),
    ("输入法", "IME → 选择“你”"),
    ("字符确定", "字符“你”"),
    ("字形信息", "font → glyph 你"),
    ("显示", "pixels → 你"),
]

def flow_column(title, items, accent):
    blocks = []
    for label, value in items:
        blocks.append(f"""
        <div style="border:2px solid {accent}; border-radius:8px; padding:10px 12px; margin:7px 0; background:#fff;">
          <div style="font-size:13px;color:#666;">{label}</div>
          <div style="font-size:18px;font-weight:700;color:#222;">{value}</div>
        </div>
        """)
    arrow = '<div style="text-align:center;font-size:22px;color:#6251B1;">↓</div>'
    return f"""
    <div style="width:46%;vertical-align:top;">
      <div style="font-size:23px;font-weight:800;margin-bottom:12px;">{title}</div>
      {arrow.join(blocks)}
    </div>
    """

html = f"""
<div style="display:flex; justify-content:space-between; gap:28px; font-family:-apple-system,BlinkMacSystemFont,'Segoe UI','Microsoft YaHei',sans-serif;">
  {flow_column("英文字符 A", english, "#6251B1")}
  {flow_column("中文字符 你", chinese, "#FF0000")}
</div>
<div style="margin-top:20px; padding:14px 18px; border-left:5px solid #6251B1; background:#f7f5ff; font-size:18px;">
  中文输入多了 <b>输入码 → IME → 选字</b>；英文和中文最终都需要 <b>字形信息 → 像素</b>。
</div>
"""

display(HTML(html))

## glyph 字形码

In [ ]:
ch = "永"

print(ch)
print(ord(ch))
print(hex(ord(ch)))

In [ ]:
print(f"Character: {ch}")
print(f"Unicode:   U+{ord(ch):04X}")

In [ ]:
# same character, different glyphs
import matplotlib.pyplot as plt

fonts = [
    "Songti SC",
    "Heiti SC",
    "Kaiti SC",
]

fig, ax = plt.subplots(figsize=(9, 3))

for i, font in enumerate(fonts):
    ax.text(
        i,
        0.6,
        "永",
        fontsize=80,
        fontfamily=font,
        ha="center"
    )

    ax.text(
        i,
        0.15,
        font,
        fontsize=12,
        ha="center"
    )

ax.set_xlim(-0.5, len(fonts) - 0.5)
ax.set_ylim(0, 1)
ax.axis("off")

plt.show()

In [ ]:
from PIL import Image, ImageDraw, ImageFont
from matplotlib import font_manager
import matplotlib.pyplot as plt

# font_path = font_manager.findfont("PingFang SC")
font_path = font_manager.findfont("Songti SC")

# 2. Create a small image
size = 48
img = Image.new("L", (size, size), 255)

# 3. Load the font
font = ImageFont.truetype(font_path, 36)

# 4. Draw one Chinese character
draw = ImageDraw.Draw(img)
draw.text((6, 2), "永", font=font, fill=0)

# 5. Enlarge it WITHOUT smoothing
scale = 10
big = img.resize(
    (size * scale, size * scale),
    Image.Resampling.NEAREST
)

# 6. Display
plt.figure(figsize=(6, 6))
plt.imshow(big, cmap="gray")
plt.axis("off")
plt.show()

In [ ]:
from PIL import Image, ImageDraw, ImageFont
from matplotlib import font_manager

# Try to find a Chinese font on macOS
font_path = font_manager.findfont(
    font_manager.FontProperties(family="PingFang SC")
)

char = "永"
img_size = 24
font_size = 20

# 1. Create a white image
img = Image.new("L", (img_size, img_size), 255)

# 2. Load font
font = ImageFont.truetype(font_path, font_size)

# 3. Draw the character
draw = ImageDraw.Draw(img)
draw.text((2, 0), char, font=font, fill=0)

# 4. Convert to 0/1
#    1 = black (stroke), 0 = white (background)
pixels = []
threshold = 200

for y in range(img_size):
    row = []
    for x in range(img_size):
        value = img.getpixel((x, y))
        bit = 1 if value < threshold else 0
        row.append(bit)
    pixels.append(row)

# 5. Print result
for row in pixels:
    print(" ".join(str(v) for v in row))

In [ ]:
from PIL import Image, ImageDraw, ImageFont
from matplotlib import font_manager
import matplotlib.pyplot as plt

# Find a Chinese font
font_path = font_manager.findfont(
    font_manager.FontProperties(family="PingFang SC")
)

char = "永"
img_size = 24
font_size = 20
threshold = 200

# Create grayscale image
img = Image.new("L", (img_size, img_size), 255)
font = ImageFont.truetype(font_path, font_size)
draw = ImageDraw.Draw(img)
draw.text((2, 0), char, font=font, fill=0)

# Convert to binary 0/1
data = []
for y in range(img_size):
    row = []
    for x in range(img_size):
        value = img.getpixel((x, y))
        bit = 1 if value < threshold else 0
        row.append(bit)
    data.append(row)

# Plot
plt.figure(figsize=(8, 8))
plt.imshow(data, cmap="gray_r", interpolation="nearest")

# Draw the 0/1 text in each cell
for y in range(img_size):
    for x in range(img_size):
        plt.text(x, y, str(data[y][x]), ha="center", va="center", fontsize=7)

plt.xticks(range(img_size))
plt.yticks(range(img_size))
plt.grid(True)
plt.title(f'Glyph "{char}" shown as 0/1 pixels')
plt.show()

---

# Q13 乱码揭秘

In [10]:
text = "你好"
utf8_bytes = text.encode("gb18030")
correct = utf8_bytes.decode("gb18030")
# wrong = utf8_bytes.decode("utf-8")

print("原文              :", text)
print("GB18030 bytes       :", utf8_bytes.hex(" ").upper())
print("按 GB18030 正确解码 :", correct)

原文              : 你好
GB18030 bytes       : C4 E3 BA C3
按 GB18030 正确解码 : 你好


In [11]:
text = "你好"
utf8_bytes = text.encode("utf-8")
correct = utf8_bytes.decode("utf-8")
wrong = utf8_bytes.decode("gb18030")

print("原文              :", text)
print("UTF-8 bytes       :", utf8_bytes.hex(" ").upper())
print("按 UTF-8 正确解码 :", correct)
print("按 GBK 错误解码   :", wrong)

原文              : 你好
UTF-8 bytes       : E4 BD A0 E5 A5 BD
按 UTF-8 正确解码 : 你好
按 GBK 错误解码   : 浣犲ソ


In [12]:
print("同一批 bytes：")
print(utf8_bytes.hex(" ").upper())
print()
print("UTF-8 →", utf8_bytes.decode("utf-8"))
print("GB18030   →", utf8_bytes.decode("gb18030"))

同一批 bytes：
E4 BD A0 E5 A5 BD

UTF-8 → 你好
GB18030   → 浣犲ソ


---

# 课前检查

In [13]:
import sys
print("Python:", sys.version.split()[0])
assert "A".encode("ascii") == b"A"
assert "中".encode("gb18030").hex().upper() == "D6D0"
assert "你".encode("utf-8").hex().upper() == "E4BDA0"
print("Encoding checks: OK")

Python: 3.11.16
Encoding checks: OK
